# Xây dựng một đồ thị quan hệ Người dùng - Người dùng (User-User Graph) dựa trên việc họ có cùng tương tác với các sản phẩm giống nhau.

## Bước này tạo files:

- user_graph_dict.npy (xài sentence_transformer)


In [1]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from tqdm import tqdm

In [2]:
PATH = "../data/2023"

In [3]:
os.makedirs(os.path.join(PATH, "step5_user_graph_dict"), exist_ok=True)

In [4]:
train_df = pd.read_csv(os.path.join(PATH, "step2_train_test_split", "df_inter.label.inter"), sep="\t")

In [5]:
uid_field = "userID"
iid_field = "itemID"

# Xác định số lượng User và Item dựa trên ID lớn nhất (Tránh lỗi Index Out of Bounds)
num_user = int(train_df[uid_field].max() + 1)
num_item = int(train_df[iid_field].max() + 1)
print(f"📊 Dataset: {num_user} Users, {num_item} Items")

# Chỉ lấy dữ liệu train (x_label == 0) để tạo ma trận user-item
train_df = train_df[train_df["x_label"] == 0].copy()
train_data = train_df[[uid_field, iid_field]].to_numpy()

📊 Dataset: 150673 Users, 35997 Items


In [6]:
# XÂY DỰNG ĐỒ THỊ USER-USER (SPARSE APPROACH)
print("🏗️  Đang xây dựng ma trận tương tác thưa...")
u_ids = train_data[:, 0]
i_ids = train_data[:, 1]
values = np.ones(len(train_data))

# Tạo ma trận User-Item
user_item_matrix = csr_matrix((values, (u_ids, i_ids)), shape=(num_user, num_item))

print("🤝 Đang tính toán độ tương đồng (User-User dot product)...")
# Ma trận User-User = UI * UI_transpose
# Kết quả là số lượng item chung giữa mỗi cặp user
user_graph_sparse = user_item_matrix.dot(user_item_matrix.T)

# Loại bỏ tự tương tác (đường chéo chính = 0)
user_graph_sparse.setdiag(0)
user_graph_sparse.eliminate_zeros()

🏗️  Đang xây dựng ma trận tương tác thưa...
🤝 Đang tính toán độ tương đồng (User-User dot product)...


In [7]:
# LỌC TOP-K HÀNG XÓM
K_MAX = 200
user_graph_dict = {}

print(f"🔝 Đang trích xuất Top-{K_MAX} hàng xóm cho từng User...")
for i in tqdm(range(num_user), desc="Processing Graph"):
    row_data = user_graph_sparse.getrow(i)

    if row_data.nnz > 0:
        indices = row_data.indices
        weights = row_data.data

        # Lấy Top-K nhanh bằng argpartition (không cần sort toàn bộ)
        if len(weights) > K_MAX:
            # Tìm vị trí của K giá trị lớn nhất
            top_k_idx = np.argpartition(weights, -K_MAX)[-K_MAX:]
            # Sắp xếp lại 200 cái đó theo thứ tự giảm dần cho chuẩn
            sorted_rel_idx = top_k_idx[np.argsort(weights[top_k_idx])[::-1]]

            user_graph_dict[i] = [
                indices[sorted_rel_idx].tolist(),
                weights[sorted_rel_idx].tolist(),
            ]
        else:
            # Nếu ít hơn K_MAX thì lấy hết và sắp xếp
            sorted_idx = np.argsort(weights)[::-1]
            user_graph_dict[i] = [
                indices[sorted_idx].tolist(),
                weights[sorted_idx].tolist(),
            ]
    else:
        # Trường hợp user không có hàng xóm nào
        user_graph_dict[i] = [[], []]

🔝 Đang trích xuất Top-200 hàng xóm cho từng User...


Processing Graph:   0%|          | 0/150673 [00:00<?, ?it/s]

Processing Graph: 100%|██████████| 150673/150673 [00:16<00:00, 9117.85it/s] 


In [8]:
output_path = os.path.join(PATH, "step5_user_graph_dict", "user_graph_dict.npy")

# Lưu dưới dạng file .npy cho MMRec load nhanh
np.save(output_path, user_graph_dict, allow_pickle=True)

print("\n" + "=" * 30)
print(f"✅ HOÀN THÀNH!")
print(f"📂 Đồ thị đã lưu tại: {output_path}")
print(f"🚀 Bạn đã có thể dùng file này để train model.")


✅ HOÀN THÀNH!
📂 Đồ thị đã lưu tại: ../data/2023\step5_user_graph_dict\user_graph_dict.npy
🚀 Bạn đã có thể dùng file này để train model.
